# Video Category Pipeline
- Reference data
- One-time extraction
- Raw JSON → Volume
- Auto Loader → Bronze Delta

In [0]:
# %pip install google-api-python-client python-dotenv

# %restart_python

## 1. Create Volume and `incoming` folder

In [0]:
%sql
-- volume; vol_video_category
CREATE VOLUME IF NOT EXISTS youtube_content_intelligence.bronze.vol_video_category;

In [0]:
# folder; incoming
dbutils.fs.mkdirs("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/")

## 2. Extract video categories data from YouTube API

In [0]:
from src.extraction.video_category import extract_video_categories

categories = extract_video_categories()

## 3. Write raw video category JSON to Volume

In [0]:
import json

volume_path = "/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json"

with open(volume_path, "w") as file:
    json.dump(categories, file, indent=2)

## 4. Verify raw video category JSON

In [0]:
display(dbutils.fs.ls("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/"))

## 5. Load raw JSON with Auto Loader

In [0]:
# Inspect the raw video_category JSON structure

display(dbutils.fs.head(
    "/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json",
    10000
))

In [0]:
# Define the schema for the video_category API response

from pyspark.sql.types import StructType, StructField, StringType, BooleanType, ArrayType

schema = StructType([
    StructField("kind", StringType(), True),
    StructField("etag", StringType(), True),
    StructField("items", ArrayType(
        StructType([
            StructField("kind", StringType(), True),
            StructField("etag", StringType(), True),
            StructField("id", StringType(), True),
            StructField("snippet", StructType([
                StructField("title", StringType(), True),
                StructField("assignable", BooleanType(), True),
                StructField("channelId", StringType(), True)
            ]), True)
        ])
    ), True)
])

In [0]:
# Read video_category JSON from the Volume with Auto Loader

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/youtube_content_intelligence/bronze/vol_video_category/schema/"
    )
    .schema(schema)
    .load("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/")
)

In [0]:
df.printSchema()

In [0]:
# Write video_category data to the Bronze Delta table

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/youtube_content_intelligence/bronze/vol_video_category/checkpoint/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("youtube_content_intelligence.bronze.brz_video_category")
)

In [0]:
# Verify the video_category Bronze Delta table

display(spark.table("youtube_content_intelligence.bronze.brz_video_category"))

### Clean Ups

In [0]:
%sql
DROP TABLE IF EXISTS youtube_content_intelligence.bronze.brz_video_category;

In [0]:
dbutils.fs.rm("/Volumes/youtube_content_intelligence/bronze/vol_video_category/checkpoint/", True)

In [0]:
dbutils.fs.rm("/Volumes/youtube_content_intelligence/bronze/vol_video_category/schema/", True)

### Retry experiment

In [0]:
# Test Auto Loader schema inference for video_category

df_inferred = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/youtube_content_intelligence/bronze/vol_video_category/schema_inferred/")
    .load("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/")
)

df_inferred.printSchema()

In [0]:
display(df_inferred.select("_rescued_data"))

In [0]:
display(df_inferred.select("_rescued_data"), checkpointLocation="/Volumes/youtube_content_intelligence/bronze/vol_video_category/checkpoint_inferred/")

In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json")
)

df.printSchema()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("youtube_content_intelligence.bronze.brz_video_category")

## Validate Bronze table